In [74]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')  

In [75]:
data_dir = os.path.join('..', 'data', 'adult') 
output_dir = os.path.join('..', 'data', 'processed')

In [76]:
input_files = [
    os.path.join(data_dir, '2009-adult.xls'),
    os.path.join(data_dir, '2010-adult.xls'),
    os.path.join(data_dir, '2011-adult.xls'),
    os.path.join(data_dir, '2012-adult.xls'),
    os.path.join(data_dir, '2013-adult.xlsx'),
    os.path.join(data_dir, '2014-adult.xlsx'),
    os.path.join(data_dir, '2015-adult.xlsx'),
    os.path.join(data_dir, '2016-adult.xlsx')
]

In [77]:
raw_dir = os.path.join(output_dir, 'raw')
measure_dir = os.path.join(output_dir, 'measure')

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(measure_dir, exist_ok=True)

In [78]:
# Cleaning
def clean_number(val):
    if pd.isna(val):
        return np.nan
    if not isinstance(val, str):
        try:
            return float(val)
        except:
            return val
    # Remove $ and commas
    val = re.sub(r'[$,]', '', str(val))
    # Handle percentages
    if '%' in val:
        val = val.replace('%', '')
        try:
            return float(val) / 100
        except:
            return val
    # Try to convert to float
    try:
        return float(val)
    except:
        return val

In [79]:
def process_adult_data(file_path):
    df = pd.read_excel(file_path)
    
    year = os.path.basename(file_path).split('-')[0]
    
    # Handle different year formats  
    if year in ['2012', '2013', '2014']:
        measures_df = df.iloc[:88]  
        raw_df = df.iloc[89:142]  # Only keep up to line 142
    elif year in ['2015', '2016']:
        measures_df = df.iloc[:89]
        raw_df = df.iloc[90:145]  # Only keep up to line 90
    else:
        # Processing for 2009-2011
        measures_df = df.iloc[:70]   
        raw_df = df.iloc[72:115]
    
    measures_df = measures_df.T
    raw_df = raw_df.T
    
    measures_df.columns = measures_df.iloc[0]
    measures_df = measures_df.drop(measures_df.index[0])
    raw_df.columns = raw_df.iloc[0]
    raw_df = raw_df.drop(raw_df.index[0])
    
    measures_df = measures_df.map(clean_number)
    raw_df = raw_df.map(clean_number)
    
    measures_df = measures_df.reset_index().rename(columns={'index': 'County'})
    raw_df = raw_df.reset_index().rename(columns={'index': 'County'})
    
    measures_df = measures_df[~measures_df['County'].str.contains('California|Unnamed', case=False, na=False)]
    raw_df = raw_df[~raw_df['County'].str.contains('California|Unnamed', case=False, na=False)]
    
    measures_df = measures_df.dropna(axis=1, how='all')
    raw_df = raw_df.dropna(axis=1, how='all')
    
    if year in ['2013', '2014']:
        measures_df = measures_df.iloc[:58]
        raw_df = raw_df.iloc[:58]
    
    print(f"measures_df shape {measures_df.shape}\n")
    print(f"measures_df for the year {year}: {measures_df.head()}\n")
    print(f"raw_df shape {raw_df.shape}\n")
    print(f"raw_df for the year {year}: {raw_df.head()}\n")
    
    return measures_df, raw_df

In [80]:
for file_path in input_files:
    year = os.path.basename(file_path).split('-')[0]
    
    measures_df, raw_df = process_adult_data(file_path)
    
    measures_output = os.path.join(measure_dir, f'{year}_measures_cleaned.csv')
    raw_output = os.path.join(raw_dir, f'{year}_raw_numbers_cleaned.csv')
        
    measures_df.to_csv(measures_output, index=False)
    raw_df.to_csv(raw_output, index=False)

measures_df shape (58, 69)

measures_df for the year 2009: Measures     County  Imprisonments per 1,000 annual adult felony arrests  \
0           Alameda                                         284.052533     
1            Alpine                                         200.000000     
2            Amador                                         508.009153     
3             Butte                                         704.591837     
4         Calaveras                                         267.898383     

Measures  County imprisonment rate as percent of state average (=100%), arrest based  \
0                                                  0.680107                            
1                                                  0.478860                            
2                                                  1.216327                            
3                                                  1.687005                            
4                                           